# PySpark + venv setup for the manual smoke notebook

One-time setup helper for `notebook_overall.ipynb` (the actual Qualifire walkthrough).
Does **not** run any Qualifire code itself.

Goal: end up with a Jupyter kernel where:

- `sys.executable` points at a project-local virtualenv
  (`.venv/bin/python` inside the qualifire repo),
- `import pyspark` works,
- `import qualifire` works (the repo is editable-installed),
- nothing on `sys.path` lives under `~/Downloads/` (macOS Privacy
  blocks reading that folder for many GUI-launched processes, which
  silently breaks both `import` discovery and `pip` itself).

Run sections **1 → 2 → 3** once, then switch to `notebook_overall.ipynb`.
Section **4** is a runtime fallback for any kernel where pyspark isn't
yet installed.

## 1. Create the venv (run in a terminal, not in this notebook)

Homebrew Python 3.11 is *externally managed* (PEP 668), so installing into
it directly requires `--break-system-packages` and pollutes a system-managed
interpreter. A project-local venv avoids both problems.

```bash
cd /Users/amitranjan/IdeaProjects/qualifire
python3.11 -m venv .venv

.venv/bin/pip install --upgrade pip
.venv/bin/pip install -e '.[all]' 'pyspark==3.5.0' ipykernel
```

`-e '.[all]'` editable-installs the qualifire package plus every optional
extra (anomaly, forecast, dev, …). `pyspark==3.5.0` matches whatever Spark
distro you may also have on disk — keep them aligned to avoid driver/worker
version drift.

If you previously created the venv elsewhere (e.g.
`/Users/amitranjan/IdeaProjects/venv/qualifire/`), don't try to `mv` it —
venvs hardcode their own absolute path in `bin/activate`, every console
script's shebang, and `pyvenv.cfg`. Recreate it fresh as above and delete
the old one with `rm -rf`.

## 2. Point VS Code at the new venv

VS Code's notebook UI does **not** read `~/Library/Jupyter/kernels/`. It
discovers Python interpreters directly. So `python -m ipykernel install`
is not required for VS Code (only for classic Jupyter / JupyterLab).

1. Open `notebook_overall.ipynb` in VS Code.
2. Top-right **kernel selector** → **Select Another Kernel… → Python
   Environments…**.
3. Pick `.venv (Python 3.11)` from `/Users/amitranjan/IdeaProjects/qualifire`.
   If it isn't listed, choose **Enter interpreter path…** and paste:
   `/Users/amitranjan/IdeaProjects/qualifire/.venv/bin/python`.
4. Also run **Command Palette → Python: Select Interpreter** and pick the
   same `.venv`. That makes the integrated terminal, Pylance, and the test
   runner use the same env as the notebook.

VS Code stores the kernel choice in the notebook's `metadata.kernelspec`.
If you ever want to force a fresh prompt (e.g. switching machines), delete
that block:

```python
import json
p = '/Users/amitranjan/IdeaProjects/qualifire/tests/manual/notebook_overall.ipynb'
nb = json.load(open(p))
nb.get('metadata', {}).pop('kernelspec', None)
json.dump(nb, open(p, 'w'), indent=1)
```

If you also want the kernel registered for non-VS-Code Jupyter
(`jupyter notebook`, `jupyter lab`):

```bash
/Users/amitranjan/IdeaProjects/qualifire/.venv/bin/python -m ipykernel install \
    --user --name qualifire --display-name 'Python (qualifire)'
```

`ipykernel install` always overwrites the kernelspec of the same `--name`,
so re-run that line whenever the venv path changes.

## 3. Verify the kernel is what you think it is

Open this notebook with the new kernel selected and run the cells below.
Expectations:

- `sys.executable` ends in `/qualifire/.venv/bin/python`,
- `pyspark.__file__` is under `…/.venv/lib/python3.11/site-packages/pyspark`,
- `qualifire.__file__` is under your repo (editable install).

In [ ]:
import os, sys
print('sys.executable :', sys.executable)
print('sys.prefix     :', sys.prefix)
print('VIRTUAL_ENV    :', os.environ.get('VIRTUAL_ENV'))
print('PYTHONPATH     :', os.environ.get('PYTHONPATH'))
print()
print('first 5 sys.path entries:')
for p in sys.path[:5]:
    print('   ', p)

In [ ]:
import importlib
for mod in ('pyspark', 'qualifire'):
    try:
        m = importlib.import_module(mod)
        print(f'{mod:10s} OK  version={getattr(m, "__version__", "?")}  path={m.__file__}')
    except Exception as e:
        print(f'{mod:10s} FAIL {type(e).__name__}: {e}')

## 4. Runtime fallback — Downloads-folder gotcha + on-the-fly pip install

Use this **only** if the verification above failed. Two things commonly go
wrong on macOS:

**A. `~/Downloads/...` on `sys.path` poisons everything.** macOS Privacy &
Security blocks GUI-launched processes (VS Code, Code Helper) from reading
Downloads / Documents / Desktop. If your shell `PYTHONPATH` includes a
Spark distro under Downloads, the kernel inherits it, and *any* package
discovery (including pip's) crashes with
`PermissionError: Operation not permitted: '/Users/.../Downloads/...'`.
Symptom: `import pyspark` reports "No module named 'pyspark'" even though
the package is installed, AND `pip install` / `pip show` crash with the
same `PermissionError` deep inside pip.

Permanent fixes (any one is enough):

1. Move the Spark distro out of Downloads (`mv ~/Downloads/apps/spark-3.5.0 ~/apps/`).
2. Remove the `export PYTHONPATH=…/Downloads/…` line from your shell rc.
3. Grant **Full Disk Access** (or just *Downloads Folder*) to VS Code in
   System Settings → Privacy & Security → Files & Folders.

**B. The kernel's interpreter just doesn't have pyspark yet.** Install it
via the *running* interpreter (`sys.executable -m pip install`) so it
absolutely lands where the kernel will look — bare `pip` from a shell can
resolve to a different env and lie to you.

The cell below combines both: scrub Downloads from `sys.path` and
`PYTHONPATH`, then pip-install pyspark, then re-import.

In [ ]:
import os, sys, subprocess

# A. Scrub Downloads-rooted entries (macOS Privacy blocks them).
before = len(sys.path)
sys.path[:] = [p for p in sys.path if '/Downloads/' not in p]
os.environ.pop('PYTHONPATH', None)
print(f'sys.path: kept {len(sys.path)}/{before} entries; PYTHONPATH unset')

# B. Install pyspark into THIS interpreter, retrying with PEP-668 bypass.
PYSPARK_VERSION = '3.5.0'
for flags in [[], ['--user'], ['--break-system-packages'], ['--user', '--break-system-packages']]:
    cmd = [sys.executable, '-m', 'pip', 'install', f'pyspark=={PYSPARK_VERSION}', *flags]
    print('  $', ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0:
        print('  ok')
        break
    print('  failed (rc=', r.returncode, '):', (r.stderr or r.stdout).strip().splitlines()[-2:])
else:
    raise RuntimeError('pip install failed under all flag combos — see stderr above')

# Drop any cached negative imports before retrying.
for m in [k for k in list(sys.modules) if k == 'pyspark' or k.startswith('pyspark.')]:
    sys.modules.pop(m, None)

import pyspark
print('pyspark', pyspark.__version__, '@', pyspark.__file__)

Once section 3 prints `OK` for both `pyspark` and `qualifire`, you're done.
Switch to `notebook_overall.ipynb` (with the same kernel selected) and run it
top-to-bottom.